# 04. Feature engineering

**Principle:** use only information known at posting time.

**Excluded on purpose**
- Leak the label: views, likes, comments, shares, saves, downloads, engagement_rate, stats_updated_at
- Decided after posting: not_recommended_for_fyp, sound_muted
- Nearly empty: width, height, is_ai_generated

**Features built:** hour and weekday of posting, country (top 10, others grouped as OTHER),
language, type (video or photo), duration, image count, caption length and word count,
number of hashtags, mentions and on-screen text, ad / shop / branded flags,
and sound popularity.

**Leakage control:** sound popularity and the country list are computed on the train set only.
For train rows, the video itself is removed from its own sound count.

In [6]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
P = ROOT / "data" / "processed"

train = pl.read_parquet(P / "train_labeled.parquet")
val   = pl.read_parquet(P / "validation_labeled.parquet")
test  = pl.read_parquet(P / "test_labeled.parquet")
print(train.shape, val.shape, test.shape)

(27597, 37) (14398, 37) (17999, 37)


In [7]:
d = train
checks = {
    "country filled":     (d["country"] != "").mean(),
    "language known":     (d["language"] != "un").mean(),
    "caption filled":     (d["caption"] != "").mean(),
    "has hashtags":       (d["hashtags"].list.len() > 0).mean(),
    "has mentions":       (d["mentions"].list.len() > 0).mean(),
    "has on_screen_text": (d["on_screen_text"].list.len() > 0).mean(),
    "duration > 0":       (d["duration_ms"] > 0).mean(),
    "is_ad":              d["is_ad"].mean(),
    "is_shop_video":      d["is_shop_video"].mean(),
}
for k, v in checks.items():
    print(f"{k:20s} {v:.1%}")

print(d["type"].value_counts())
print(d.group_by("country").agg(pl.len().alias("n"), pl.col("is_viral").mean().alias("viral_rate")).sort("n", descending=True).head(10))


country filled       100.0%
language known       45.0%
caption filled       72.1%
has hashtags         54.9%
has mentions         7.9%
has on_screen_text   24.5%
duration > 0         87.6%
is_ad                1.0%
is_shop_video        2.5%
shape: (2, 2)
┌───────┬───────┐
│ type  ┆ count │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ photo ┆ 3417  │
│ video ┆ 24180 │
└───────┴───────┘
shape: (10, 3)
┌─────────┬───────┬────────────┐
│ country ┆ n     ┆ viral_rate │
│ ---     ┆ ---   ┆ ---        │
│ str     ┆ u32   ┆ f64        │
╞═════════╪═══════╪════════════╡
│ US      ┆ 21785 ┆ 0.094744   │
│ GB      ┆ 866   ┆ 0.116628   │
│ ID      ┆ 748   ┆ 0.074866   │
│ PK      ┆ 488   ┆ 0.112705   │
│ FR      ┆ 369   ┆ 0.149051   │
│ TH      ┆ 257   ┆ 0.101167   │
│ GH      ┆ 171   ┆ 0.05848    │
│ PH      ┆ 155   ┆ 0.051613   │
│ VN      ┆ 143   ┆ 0.125874   │
│ TR      ┆ 132   ┆ 0.181818   │
└─────────┴───────┴────────────┘


In [10]:
# Sound popularity and country list are computed on TRAIN only, to avoid leakage.
sound_counts = train.group_by("sound_id").len().rename({"len": "sound_freq"})
top_countries = (train.group_by("country").len()
                      .sort("len", descending=True).head(10)["country"].to_list())

def build(d, is_train=False):
    out = d.select([
        "video_id", "posted_at", "split", "is_viral", "caption",
        pl.col("posted_at").dt.hour().alias("hour"),
        pl.col("posted_at").dt.weekday().alias("weekday"),
        pl.col("posted_at").dt.month().alias("month"),
        pl.when(pl.col("country").is_in(top_countries))
          .then(pl.col("country")).otherwise(pl.lit("OTHER")).alias("country"),
        "language", "type", "duration_ms", "image_count",
        pl.col("caption").str.len_chars().alias("caption_len"),
        pl.when(pl.col("caption") == "").then(0)
          .otherwise(pl.col("caption").str.split(" ").list.len()).alias("caption_words"),
        pl.col("hashtags").list.len().alias("n_hashtags"),
        pl.col("mentions").list.len().alias("n_mentions"),
        pl.col("on_screen_text").list.len().alias("n_onscreen"),
        "is_ad", "is_shop_video",
        (pl.col("branded_content") != "").cast(pl.UInt8).alias("is_branded"),
        "sound_id",
    ]).join(sound_counts, on="sound_id", how="left")
    out = out.with_columns(pl.col("sound_freq").fill_null(0))
    if is_train:  # remove the video itself from its own count
        out = out.with_columns((pl.col("sound_freq") - 1).alias("sound_freq"))
    return out.drop("sound_id")

f_train = build(train, is_train=True)
f_val   = build(val)
f_test  = build(test)

for name, f in [("train", f_train), ("validation", f_val), ("test", f_test)]:
    f.write_parquet(P / f"{name}_features.parquet")

print(f_train.shape)
f_train.head(3)

(27597, 22)


video_id,posted_at,split,is_viral,caption,hour,weekday,month,country,language,type,duration_ms,image_count,caption_len,caption_words,n_hashtags,n_mentions,n_onscreen,is_ad,is_shop_video,is_branded,sound_freq
u64,"datetime[ms, UTC]",str,u8,str,i8,i8,i8,str,str,str,u32,u8,u32,u32,u32,u32,u32,u8,u8,u8,u32
7454735578760744199,2025-01-01 00:21:32 UTC,"""train""",0,"""#ကပြားမလေး🇬🇦🇧🇫✌️🥰 #မြင်တာများပ…",0,3,1,"""OTHER""","""un""","""video""",15000,0,72,4,3,0,0,0,0,0,0
7454737156888595718,2025-01-01 00:27:35 UTC,"""train""",0,"""""Happy New Year from Us🎉✨ We'…",0,3,1,"""GH""","""en""","""video""",7000,0,425,77,7,0,0,0,0,0,1
7454764036325084462,2025-01-01 02:11:52 UTC,"""train""",0,"""adios 2024 bienvenido 2025""",2,3,1,"""US""","""es""","""video""",53000,0,26,4,0,0,1,0,0,0,0


In [9]:
import polars as pl
d = pl.concat([train, val, test])

print("rows:", d.height)
print("duplicate video_id:", d.height - d["video_id"].n_unique())
print("views == 0:", (d["views"] == 0).sum(), "| views null:", d["views"].null_count())
print("duration_ms == 0:", (d["duration_ms"] == 0).sum())
print("duration_ms max (min):", d["duration_ms"].max() / 60000)
print("posted_at null:", d["posted_at"].null_count())
print(d["type"].value_counts())
print("nulls per column with nulls:")
nulls = d.null_count().transpose(include_header=True, column_names=["nulls"])
print(nulls.filter(pl.col("nulls") > 0))
print(d.group_by("split").agg(pl.col("posted_at").min().alias("from"), pl.col("posted_at").max().alias("to")))

rows: 59994
duplicate video_id: 0
views == 0: 220 | views null: 0
duration_ms == 0: 4268
duration_ms max (min): 59.983333333333334
posted_at null: 0
shape: (2, 2)
┌───────┬───────┐
│ type  ┆ count │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ photo ┆ 4268  │
│ video ┆ 55726 │
└───────┴───────┘
nulls per column with nulls:
shape: (3, 2)
┌─────────────────────────┬───────┐
│ column                  ┆ nulls │
│ ---                     ┆ ---   │
│ str                     ┆ u32   │
╞═════════════════════════╪═══════╡
│ is_ai_generated         ┆ 59487 │
│ not_recommended_for_fyp ┆ 26878 │
│ engagement_rate         ┆ 220   │
└─────────────────────────┴───────┘
shape: (3, 3)
┌────────────┬─────────────────────────┬─────────────────────────┐
│ split      ┆ from                    ┆ to                      │
│ ---        ┆ ---                     ┆ ---                     │
│ str        ┆ datetime[ms, UTC]       ┆ datetime[ms, UTC]       │
╞════════════╪═════════════════════════╪════